Notebook 6 — Train, tune, evaluate
------

In [4]:
import mlflow
import mlflow.sklearn


In [2]:
print(mlflow.__version__)

3.16.0


In [8]:
mlflow.set_experiment("Olist Delivery Prediction")

<Experiment: artifact_location=('file:d:/Zain MS/Programming/Machine Learning/Qafza_MLOps/closed '
 'Tasks/Task_3/olist-mlops/notebooks/mlruns/1'), creation_time=1789505844072, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789505844072, lifecycle_stage='active', name='Olist Delivery Prediction', tags={}, trace_location=None, workspace='default'>

In [ ]:
# workflow should be:  
# Notebook 5  
# Raw Train / Validation / Test  
#         ↓  
# Feature Engineering + Preprocessing  
#         ↓  
# X_train_final  
# X_val_final  
# X_test_final  
#         ↓  
# Notebook 6  
# Baseline  
#         ↓  
# Train Models  
#         ↓  
# Tune on Validation Set  
#         ↓  
# Select Best Model  
#         ↓  
# Touch Test Set ONCE  
#         ↓  
# Final Evaluation  
#         ↓  
# Save Model + Results  


Define the goal  
target is: is_late_label  
Where:  
1 = Late delivery    
0 = On-time delivery  


Start with a dummy baseline
----

In [5]:
# Before training a real model, establish a baseline.
# importing one baseline model and five evaluation metrics
from sklearn.dummy import (
    DummyClassifier,  # DummyClassifier creates a very simple baseline classifier.
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# These import functions to evaluate your model predictions.
# | Metric            | What it measures                                                    |
# | ----------------- | ------------------------------------------------------------------- |
# | `accuracy_score`  | Overall percentage of correct predictions                           |
# | `precision_score` | When the model predicts **late**, how often is it correct?          |
# | `recall_score`    | Out of all actual **late deliveries**, how many did the model find? |
# | `f1_score`        | Balance between precision and recall                                |
# | `roc_auc_score`   | How well the model separates late vs on-time orders                 |




In [10]:
# check what files you have in your project folders:
from pathlib import Path

for path in Path(".").rglob("*"):
    if path.is_file():
        print(path)

Notebook 1 — Read & join the tables.ipynb
Notebook 2 — Create the labels.ipynb
Notebook 3 — Train   validation   test split.ipynb
Notebook 4 — EDA (the detailed one).ipynb
Notebook 5 — Feature engineering.ipynb
Notebook 6 — Train, tune, evaluate.ipynb


In [11]:
# check what files you have in your artifacts folders:
from pathlib import Path

print("Files in artifacts folder:")

artifacts_path = Path("artifacts")

if artifacts_path.exists():
    for file in artifacts_path.iterdir():
        print(file)
else:
    print("❌ artifacts folder not found")

Files in artifacts folder:
❌ artifacts folder not found


In [6]:
import sqlite3

import pandas as pd

In [7]:
# connect to db and showing available tables inside
db_path = r"D:\2026\MLOps-Qafza-2026\database\olist.db"

conn = sqlite3.connect(db_path)

query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)
display(tables)

conn.close()

,name
0,agg_olist_order_items
1,agg_olist_order_payments
2,agg_olist_order_reviews
3,ml_orders
4,ml_orders_labled
5,olist_customers
6,olist_geolocation
7,olist_order_items
8,olist_order_payments
9,olist_order_reviews


In [ ]:
# in Notebook 6, you need to:

# Load your original train/validation/test datasets.
# Separate features (X) and target (y).
# Load preprocessor.joblib.
# Transform the datasets again to create X_train_final, X_val_final, and X_test_final.

In [8]:
# load the original train/validation/test datasets
import sqlite3

import pandas as pd

db_path = r"D:\2026\MLOps-Qafza-2026\database\olist.db"

conn = sqlite3.connect(db_path)

training_set = pd.read_sql(
    "SELECT * FROM updated_training_set",
    conn
)

validation_set = pd.read_sql(
    "SELECT * FROM validation_set",
    conn
)

testing_set = pd.read_sql(
    "SELECT * FROM testing_set",
    conn
)

conn.close()

In [9]:
print("Train:", training_set.shape)
print("Validation:", validation_set.shape)
print("Test:", testing_set.shape)

Train: (67533, 34)
Validation: (14471, 34)
Test: (14472, 34)


In [10]:
# Separate features (X) and target (y).
target = "is_late_label"

X_train = training_set.drop(columns=[target])
y_train = training_set[target]

X_val = validation_set.drop(columns=[target])
y_val = validation_set[target]

X_test = testing_set.drop(columns=[target])
y_test = testing_set[target]

In [11]:
# Load preprocessor.joblib.
# preprocessor = joblib.load(
#     "artifacts/preprocessor.joblib"
# )
from pathlib import Path

import joblib

PROJECT_ROOT = Path.cwd().parent

preprocessor_path = PROJECT_ROOT / "models" / "preprocessor.joblib"

preprocessor = joblib.load(preprocessor_path)

print("Loaded preprocessor:", preprocessor_path)
print("Feature count:", len(preprocessor.get_feature_names_out()))



Loaded preprocessor: d:\Zain MS\Programming\Machine Learning\Qafza_MLOps\closed Tasks\Task_3\olist-mlops\models\preprocessor.joblib
Feature count: 3796


In [17]:
# recreate the missing features before transforming
# check the columns expected by the preprocessor:
print(preprocessor.feature_names_in_)

['order_id' 'customer_id' 'order_status' 'order_purchase_timestamp'
 'order_approved_at' 'order_delivered_carrier_date'
 'order_delivered_customer_date' 'order_estimated_delivery_date'
 'number_of_items' 'total_freight_value' 'total_price' 'number_of_sellers'
 'number_of_products' 'number_of_payments' 'total_payment_value'
 'number_of_payment_types' 'max_payment_installments' 'purchase_year'
 'purchase_month' 'purchase_dayofweek' 'purchase_hour'
 'actual_delivery_days' 'estimated_delivery_days'
 'delivery_difference_days' 'customer_unique_id'
 'customer_zip_code_prefix' 'customer_city' 'customer_state'
 'seller_state' 'seller_zip_code_prefix' 'seller_count'
 'customer_seller_same_state' 'zip_prefix_difference' 'purchase_day'
 'approval_delay_hours']


In [ ]:
# to be replaced in following cell

# recreate missed purchase_day for all three datasets.
for df in [X_train, X_val, X_test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["purchase_day"] = (
        df["order_purchase_timestamp"].dt.dayofweek
    )

In [ ]:
# to be replaced in following cell

# recreate missed approval_delay_hours for all three datasets.
for df in [X_train, X_val, X_test]:

    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["order_approved_at"] = pd.to_datetime(
        df["order_approved_at"]
    )

    df["approval_delay_hours"] = (
        df["order_approved_at"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

In [12]:
# Modofication for Task 3

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src")
)
from features.feature_engineering import create_date_features




In [13]:
X_train = create_date_features(X_train)
X_val = create_date_features(X_val)
X_test = create_date_features(X_test)

In [14]:
from features.feature_selection import select_model_features

X_train = select_model_features(X_train, target)
X_val = select_model_features(X_val, target)
X_test = select_model_features(X_test, target)


In [15]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "actual_delivery_days",
    "delivery_difference_days",
]

print(
    "Leakage columns still present:",
    [c for c in leakage_columns if c in X_train.columns]
)

print("X_train columns:", X_train.columns.tolist())


X_train shape: (67533, 28)
X_val shape: (14471, 28)
X_test shape: (14472, 28)
Leakage columns still present: []
X_train columns: ['order_id', 'customer_id', 'order_status', 'number_of_items', 'total_freight_value', 'total_price', 'number_of_sellers', 'number_of_products', 'number_of_payments', 'total_payment_value', 'number_of_payment_types', 'max_payment_installments', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_state', 'seller_zip_code_prefix', 'seller_count', 'customer_seller_same_state', 'zip_prefix_difference', 'purchase_day', 'approval_delay_hours']


In [16]:
# preprocessor = joblib.load(
#     "artifacts/preprocessor.joblib"
# )

from pathlib import Path

import joblib

PROJECT_ROOT = Path.cwd().parent

preprocessor_path = PROJECT_ROOT / "models" / "preprocessor.joblib"

preprocessor = joblib.load(preprocessor_path)

print("Loaded preprocessor:", preprocessor_path)
print("Feature count:", len(preprocessor.get_feature_names_out()))




Loaded preprocessor: d:\Zain MS\Programming\Machine Learning\Qafza_MLOps\closed Tasks\Task_3\olist-mlops\models\preprocessor.joblib
Feature count: 3796


In [17]:
X_train_final = preprocessor.transform(X_train)

X_val_final = preprocessor.transform(X_val)

X_test_final = preprocessor.transform(X_test)


In [18]:
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)


X_train_final: (67533, 3796)
X_val_final: (14471, 3796)
X_test_final: (14472, 3796)


In [19]:
# check that all required columns exist
expected_columns = set(preprocessor.feature_names_in_)

for name, X in {
    "Train": X_train,
    "Validation": X_val,
    "Test": X_test
}.items():

    missing = expected_columns - set(X.columns)

    print(f"{name} missing columns:", missing)

Train missing columns: set()
Validation missing columns: set()
Test missing columns: set()


In [20]:
# Transform the datasets again to create X_train_final, X_val_final, and X_test_final.
X_train_final = preprocessor.transform(
    X_train
)

X_val_final = preprocessor.transform(
    X_val
)

X_test_final = preprocessor.transform(
    X_test
)

In [21]:
# verify
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

X_train_final: (67533, 3796)
X_val_final: (14471, 3796)
X_test_final: (14472, 3796)


In [22]:
# A simple baseline:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_train_final,
    y_train
)

dummy_pred = dummy_model.predict(
    X_val_final
)


In [23]:
# Evaluate it:
print("Accuracy:",
      accuracy_score(y_val, dummy_pred))

print("Precision:",
      precision_score(
          y_val,
          dummy_pred,
          zero_division=0
      ))

print("Recall:",
      recall_score(
          y_val,
          dummy_pred,
          zero_division=0
      ))

print("F1:",
      f1_score(
          y_val,
          dummy_pred,
          zero_division=0
      ))
# This gives you the minimum performance your real models should beat.


Accuracy: 0.9188722272130467
Precision: 0.0
Recall: 0.0
F1: 0.0


First real model — Logistic Regression
------

In [24]:
# Logistic Regression is a good first model because:
    # •	Fast
    # •	Simple
    # •	Easy to interpret
    # •	Good baseline for classification

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_final,
    y_train
)



,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [25]:
# Make predictions
val_pred = logistic_model.predict(
    X_val_final
)

val_proba = logistic_model.predict_proba(
    X_val_final
)[:, 1]


In [26]:
# Evaluate:
print("Accuracy:",
      accuracy_score(y_val, val_pred))

print("Precision:",
      precision_score(
          y_val,
          val_pred,
          zero_division=0
      ))

print("Recall:",
      recall_score(
          y_val,
          val_pred,
          zero_division=0
      ))

print("F1:",
      f1_score(
          y_val,
          val_pred,
          zero_division=0
      ))

print("ROC-AUC:",
      roc_auc_score(
          y_val,
          val_proba
      ))


Accuracy: 0.6709971667472877
Precision: 0.14022066198595787
Recall: 0.5954003407155025
F1: 0.22698490014612763
ROC-AUC: 0.6899049483949384


Create one reusable evaluation function
--------

In [27]:
# Instead of repeating the same code for every model:
def evaluate_model(
    model,
    X,
    y,
    model_name="Model"
):
    
    predictions = model.predict(X)
    
    probabilities = model.predict_proba(X)[:, 1]
    
    results = {
        "model": model_name,
        
        "accuracy": accuracy_score(
            y,
            predictions
        ),
        
        "precision": precision_score(
            y,
            predictions,
            zero_division=0
        ),
        
        "recall": recall_score(
            y,
            predictions,
            zero_division=0
        ),
        
        "f1": f1_score(
            y,
            predictions,
            zero_division=0
        ),
        
        "roc_auc": roc_auc_score(
            y,
            probabilities
        )
    }
    
    return results
# Now:
logistic_results = evaluate_model(
    logistic_model,
    X_val_final,
    y_val,
    "Logistic Regression"
)

logistic_results


{'model': 'Logistic Regression',
 'accuracy': 0.6709971667472877,
 'precision': 0.14022066198595787,
 'recall': 0.5954003407155025,
 'f1': 0.22698490014612763,
 'roc_auc': 0.6899049483949384}

Train another model
------

In [28]:
# A good next model for your project is:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

# Start with a simple version:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_final,
    y_train
)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [29]:
# Evaluate:
rf_results = evaluate_model(
    rf_model,
    X_val_final,
    y_val,
    "Random Forest"
)

rf_results


{'model': 'Random Forest',
 'accuracy': 0.9195632644599544,
 'precision': 1.0,
 'recall': 0.008517887563884156,
 'f1': 0.016891891891891893,
 'roc_auc': 0.7720747619033588}

Compare models
-------

In [30]:
# Create a results table:
results = pd.DataFrame([
    logistic_results,
    rf_results
])

results


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.670997,0.140221,0.595400,0.226985,0.689905
1,Random Forest,0.919563,1.000000,0.008518,0.016892,0.772075


In [ ]:
# It may look like:
# Model	                Accuracy	Precision	Recall	F1  	ROC-AUC
# Logistic Regression	...	          ...	     ...	...	    ...
# Random Forest	        ...	          ...	     ...	...	    ...

# At this stage:
# Do not touch X_test_final or y_test.

# Use only:
# Training → fit models

# Validation → compare and tune models

# Test → untouched


Tune the best model
-------

In [ ]:
# Suppose Random Forest performs better.
# You can tune it using the validation split.
# start with RandomizedSearchCV instead of testing every possible combination.
# However, one important point:
# If useed RandomizedSearchCV on the training data, it internally creates cross-validation splits. That is fine, but your separate validation set should still be used as an external check.


In [31]:
# Example:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    
    "n_estimators": [
        100,
        200,
        300
    ],
    
    "max_depth": [
        None,
        10,
        20,
        30
    ],
    
    "min_samples_split": [
        2,
        5,
        10
    ],
    
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}


In [32]:
from sklearn.ensemble import RandomForestClassifier

In [33]:
# Create the search:
rf_search = RandomizedSearchCV(
    
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    
    param_distributions=param_grid,
    
    n_iter=10,
    
    scoring="f1",
    
    cv=5,
    
    random_state=42,
    
    n_jobs=-1
)


In [34]:
# Fit only using training data:
rf_search.fit(
    X_train_final,
    y_train
)


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... ver

In [35]:
# Get the best model:
best_rf = rf_search.best_estimator_

print(
    rf_search.best_params_
)


{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None}


In [36]:
# Then evaluate on validation:
best_rf_results = evaluate_model(
    best_rf,
    X_val_final,
    y_val,
    "Tuned Random Forest"
)

best_rf_results


{'model': 'Tuned Random Forest',
 'accuracy': 0.8308340819570175,
 'precision': 0.23612261806130902,
 'recall': 0.4855195911413969,
 'f1': 0.3177257525083612,
 'roc_auc': 0.7548264719828314}

Tune the classification threshold
-------

In [37]:
# This is especially useful for your imbalanced late-delivery problem.
# By default:
# Probability ≥ 0.50 → Late
# Probability < 0.50 → On-time
# But 0.50 may not give the best F1-score.
# Try different thresholds:
import numpy as np

thresholds = np.arange(
    0.1,
    0.91,
    0.05
)

threshold_results = []

val_proba = best_rf.predict_proba(
    X_val_final
)[:, 1]

for threshold in thresholds:
    
    predictions = (
        val_proba >= threshold
    ).astype(int)
    
    threshold_results.append({
        "threshold": threshold,
        
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0
        ),
        
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0
        ),
        
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0
        )
    })


In [38]:
# Create a DataFrame:
threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results


,threshold,precision,recall,f1
0,0.10,0.081128,1.000000,0.150080
1,0.15,0.081156,1.000000,0.150128
2,0.20,0.081494,1.000000,0.150706
3,0.25,0.083500,0.996593,0.154089
4,0.30,0.092072,0.956559,0.167975
5,0.35,0.113118,0.874787,0.200332
6,0.40,0.147078,0.754685,0.246179
7,0.45,0.186774,0.630324,0.288162
8,0.50,0.236123,0.485520,0.317726
9,0.55,0.310550,0.356048,0.331746


In [39]:
# Find the best threshold:
best_threshold_row = (
    threshold_results
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

best_threshold = best_threshold_row[
    "threshold"
]

best_threshold
# This threshold should be selected using the validation set only.


np.float64(0.5500000000000002)

Final model selection
--------

In [ ]:
# At this point, you should have something like:
# Dummy Baseline
#         ↓
# Logistic Regression
#         ↓
# Random Forest
#         ↓
# Tuned Random Forest
#         ↓
# Threshold Optimization
# Choose the best model based primarily on your selected metric.
# For example:
# Main metric → F1-score

# Also inspect:
# - Precision
# - Recall
# - ROC-AUC
# - Business interpretation


Touch the test set once
-------

In [40]:
# Only now:
test_proba = best_rf.predict_proba(
    X_test_final
)[:, 1]

test_pred = (
    test_proba >= best_threshold
).astype(int)


In [41]:
# Evaluate:
test_results = {
    
    "accuracy": accuracy_score(
        y_test,
        test_pred
    ),
    
    "precision": precision_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    
    "recall": recall_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    
    "f1": f1_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    
    "roc_auc": roc_auc_score(
        y_test,
        test_proba
    )
}

test_results


{'accuracy': 0.8855030403537866,
 'precision': 0.31351351351351353,
 'recall': 0.34582623509369675,
 'f1': 0.3288780882948562,
 'roc_auc': 0.7526007164300559}

In [42]:
# You can also print a classification report:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        test_pred
    )
)


              precision    recall  f1-score   support

           0       0.94      0.93      0.94     13298
           1       0.31      0.35      0.33      1174

    accuracy                           0.89     14472
   macro avg       0.63      0.64      0.63     14472
weighted avg       0.89      0.89      0.89     14472



In [43]:
# And confusion matrix:
from sklearn.metrics import confusion_matrix

confusion_matrix(
    y_test,
    test_pred
)


array([[12409,   889],
       [  768,   406]])

Save the final trained model
--------

In [44]:
# Path("artifacts").mkdir(
#     exist_ok=True
# )
# # Save:
# joblib.dump(
#     best_rf,
#     "artifacts/final_model.joblib"
# )
from pathlib import Path

import joblib

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

final_model_path = MODELS_DIR / "final_model.joblib"

joblib.dump(
    best_rf,
    final_model_path
)

print("Saved final model:", final_model_path)



Saved final model: d:\Zain MS\Programming\Machine Learning\Qafza_MLOps\closed Tasks\Task_3\olist-mlops\models\final_model.joblib


In [45]:
# Also save the threshold:
# joblib.dump(
#     best_threshold,
#     "artifacts/classification_threshold.joblib"
# )

threshold_path = MODELS_DIR / "classification_threshold.joblib"

joblib.dump(
    best_threshold,
    threshold_path
)

print("Saved classification threshold:", threshold_path)



Saved classification threshold: d:\Zain MS\Programming\Machine Learning\Qafza_MLOps\closed Tasks\Task_3\olist-mlops\models\classification_threshold.joblib


In [ ]:
# Remember: your production prediction requires both:
# preprocessor.joblib
#         +
# final_model.joblib
#         +
# classification_threshold.joblib


In [ ]:
# The prediction flow will be:
# New Order
#     ↓
# Feature Engineering
#     ↓
# Load Preprocessor
#     ↓
# Transform
#     ↓
# Load Model
#     ↓
# Predict Probability
#     ↓
# Apply Saved Threshold
#     ↓
# Late / On-time


Save the results summary
---------

In [46]:
final_results = pd.DataFrame([
    logistic_results,
    rf_results,
    best_rf_results,
    {
        "model": "Final Test Result",
        **test_results
    }
])

final_results


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.670997,0.140221,0.595400,0.226985,0.689905
1,Random Forest,0.919563,1.000000,0.008518,0.016892,0.772075
2,Tuned Random Forest,0.830834,0.236123,0.485520,0.317726,0.754826
3,Final Test Result,0.885503,0.313514,0.345826,0.328878,0.752601


In [ ]:
# Save it:
final_results.to_csv(
    "artifacts/model_results.csv",
    index=False
)


In [ ]:
# Recommended structure for your project
# For your Olist late-delivery project, I recommend this exact progression:
# 1. DummyClassifier
#         ↓
# 2. Logistic Regression
#         ↓
# 3. Random Forest
#         ↓
# 4. Compare on Validation
#         ↓
# 5. Tune the best model
#         ↓
# 6. Optimize classification threshold
#         ↓
# 7. Final evaluation on Test
#         ↓
# 8. Save:
#    - final_model.joblib
#    - preprocessor.joblib
#    - feature_names.joblib
#    - classification_threshold.joblib
#    - model_results.csv
# The most important rule for Notebook 6 is:
# The validation set helps you choose the model. The test set tells you how well your final chosen model actually performs.
# So for the next practical step, start Notebook 6 with the Dummy Baseline, then Logistic Regression, and compare their validation results before moving to Random Forest.

